# 5d. Compare context ablations under the corrected labels

This notebook revisits the validation-based feature-family ablations after the label-attribution review in Notebook 5c.

Four reduced versions of M2 remove one context family at a time: access level, frequency encodings, expected hours or peer deviations. Each is assessed under the as-run and corrected labels across the three detector families.

**Inputs:** Notebook 2's technical table; Notebook 3's curated context and split; Notebook 4's modelling module, feature list and policy; Notebook 5's scores and ablation table; and Notebook 1's insider and organisational records.

Unlike the score-based checks in 5b and 5c, this notebook refits the ablation variants because their scores were not retained by Notebook 5. It uses the shared modelling functions and seed 42. M0 and full-M2 reference scores are reused, not refitted.

**Outputs:** `ablation_corrected.csv`, `ablation_reproduction_gate.csv` and `ablation_summary.json`.

The as-run reproduction check must pass before the corrected-label comparison proceeds. These remain validation diagnostics, not replacements for the primary test result.

*Documentation copy, 8 November 2026. Notes edited for clarity; code, saved outputs and execution metadata are unchanged. This copy has not been rerun.*


In [1]:
# ============================================================
# 0. Setup
# ============================================================
import os, glob, json, time, hashlib, warnings
import numpy as np
import pandas as pd

SEED = 42
warnings.filterwarnings("ignore")
WORK = "/kaggle/working"
T0 = time.time()

def find(pattern, required=True, what=""):
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    assert hits or not required, f"Required input not attached: {pattern} ({what})"
    return hits[0] if hits else None

P_FEATS  = find("features_cyberlog.parquet", what="NB2")
P_CTX    = find("context_curated.parquet",   what="NB3")
P_SPLIT  = find("split_definition.json",     what="NB3")
P_UTILS  = find("modelling_utils.py",        what="NB4")
P_FLIST  = find("m0_feature_list.json",      what="NB4")
P_POLICY = find("threshold_policy.json",     what="NB4")
P_SCORES = find("scores_all_models.parquet", what="NB5")
P_ABL    = find("ablation_validation.csv",   what="NB5")
P_INS    = find("insiders.parquet",          what="NB1")
P_LDAP   = find("ldap_monthly.parquet",      what="NB1")

import importlib.util
spec = importlib.util.spec_from_file_location("modelling_utils", P_UTILS)
mu = importlib.util.module_from_spec(spec); spec.loader.exec_module(mu)
UTILS_SHA = hashlib.sha256(open(P_UTILS, "rb").read()).hexdigest()[:16]

POLICY = json.load(open(P_POLICY))
DET, K = POLICY["detector"], float(POLICY["k_pct"])
DETECTORS = ["if", "ae", "ens"]
print(f"utils sha256[:16]={UTILS_SHA} | frozen policy: detector={DET}, k={K}%")

SPLIT = json.load(open(P_SPLIT))
feats = pd.read_parquet(P_FEATS)
feats["user"] = feats["user"].astype(str)
feats["day"]  = pd.to_datetime(feats["day"])
d = feats["day"]
feats["partition"] = np.select(
    [(d >= SPLIT["train_start"]) & (d <= SPLIT["train_end"]),
     (d >= SPLIT["val_start"])   & (d <= SPLIT["val_end"]),
     (d >= SPLIT["test_start"])  & (d <= SPLIT["test_end"])],
    ["train", "val", "test"], default="drop")
feats = feats[feats["partition"] != "drop"].reset_index(drop=True)
MASK = {p: (feats["partition"] == p).to_numpy() for p in ("train", "val", "test")}
assert int(feats.loc[MASK["train"], "label"].sum()) == 0, "benign-train premise violated"
print(f"spine {len(feats):,} | train {MASK['train'].sum():,} "
      f"val {MASK['val'].sum():,} test {MASK['test'].sum():,}")

utils sha256[:16]=147d19b0977c79e8 | frozen policy: detector=if, k=1.0%
spine 1,393,997 | train 592,378 val 406,070 test 395,549


## 1. Construct the corrected label view

Use the same account-based derivation as Notebook 5c: retain designated insiders and the scenario-3 insider's LDAP-resolved supervisor, and relabel the remaining as-run positive accounts under the documented bystander correction.

No account identifier is hard-coded. The derivation follows the earlier interpretation rather than independently checking each source event again.


In [2]:
# ============================================================
# 1. Corrected label set (data-driven; mirrors NB5c)
# ============================================================
ins  = pd.read_parquet(P_INS);  ins["user"] = ins["user"].astype(str)
ldap = pd.read_parquet(P_LDAP); ldap["user_id"] = ldap["user_id"].astype(str)

DESIGNATED = set(ins["user"])
s3 = ins.loc[ins["scenario"].astype(int) == 3, "user"]
assert len(s3) == 1, f"expected one scenario-3 insider, found {list(s3)}"
S3_USER  = s3.iloc[0]
SUP_NAME = ldap.loc[ldap.user_id == S3_USER, "supervisor"].dropna().iloc[0]
MASQ     = set(ldap.loc[ldap.employee_name == SUP_NAME, "user_id"])
assert MASQ, f"could not resolve supervisor '{SUP_NAME}' to a user_id"
KEEP = DESIGNATED | MASQ
print(f"scenario-3 insider {S3_USER} -> supervisor {SUP_NAME} -> {sorted(MASQ)}")

feats["label_corrected"] = ((feats.label == 1) & feats.user.isin(KEEP)).astype("int8")
assert int(feats.loc[MASK["train"], "label_corrected"].sum()) == 0

Y = {
    "as_run":    {"val": feats.loc[MASK["val"], "label"].to_numpy()},
    "corrected": {"val": feats.loc[MASK["val"], "label_corrected"].to_numpy()},
}
for n, y in Y.items():
    print(f"  {n:10s}: {int(y['val'].sum())} validation positives")

scenario-3 insider PLJ1771 -> supervisor Hedda Indira Savage -> ['HIS1706']
  as_run    : 40 validation positives
  corrected : 9 validation positives


## 2. Define the four reduced representations

Remove access level, category frequencies, expected-hours deviations or peer z-scores from the full curated block, one family at a time. Every variant retains the common technical features.

The variants correspond to the ablations in Notebook 5. They assess sensitivity to removing a complete family; they do not search every individual-feature subset.


In [3]:
# ============================================================
# 2. Frame + ablation definitions
# ============================================================
M0_FEATURES = json.load(open(P_FLIST))
ctx = pd.read_parquet(P_CTX)
ctx["user"] = ctx["user"].astype(str); ctx["day"] = pd.to_datetime(ctx["day"])
CTX_COLS = [c for c in ctx.columns if c not in ("user", "day")]
ctx[CTX_COLS] = ctx[CTX_COLS].astype("float32")

base  = feats[["user", "day"] + M0_FEATURES]
frame = base.merge(ctx, on=["user", "day"], how="inner")
assert len(frame) == len(base), "context does not cover the spine"
assert (frame["user"].values == base["user"].values).all(), "merge reordered the spine"
print(f"frame {frame.shape} | M0 {len(M0_FEATURES)} + context {len(CTX_COLS)}")

peer_cols = [c for c in CTX_COLS if c.endswith("_peer_z")]
ABLATE = {
    "no_access_level": ["access_level"],
    "no_cat_freq":     ["role_freq", "dept_freq"],
    "no_hours":        ["early_start_dev", "late_end_dev", "outside_expected_hours"],
    "no_peer_z":       peer_cols,
}
for t, dcols in ABLATE.items():
    print(f"  {t:16s} drops {len(dcols):2d} -> {len(M0_FEATURES)+len(CTX_COLS)-len(dcols)} features")

frame (1393997, 61) | M0 41 + context 18
  no_access_level  drops  1 -> 58 features
  no_cat_freq      drops  2 -> 57 features
  no_hours         drops  3 -> 56 features
  no_peer_z        drops 12 -> 47 features


## 3. Refit the ablation variants

One call to `fit_detectors` for each variant fits the two base detectors and supplies their ensemble scores through the shared procedure. All fitting remains on training data; the reported ablation comparison uses validation data.

The reconstruction model retains the 40-iteration limit. Its printed `n_iter_` helps identify runs that reached that limit. Reaching the cap does not establish convergence or the best attainable performance of the architecture.


In [4]:
# ============================================================
# 3. Refit each ablation variant (shared modelling_utils, seed=42)
# ============================================================
SV = {}   # variant -> {detector: validation score vector}
AE_ITERS = {}
for tag, drop in ABLATE.items():
    t = time.time()
    cols = M0_FEATURES + [c for c in CTX_COLS if c not in drop]
    X = mu.prepare_matrix(frame, cols)
    models = mu.fit_detectors(X[MASK["train"]], seed=SEED)
    S = mu.score_detectors(models, X)
    SV[tag] = {det: S[det][MASK["val"]] for det in DETECTORS}
    AE_ITERS[tag] = int(models["ae"].n_iter_)
    ae_note = " (iteration-capped, not converged)" if models["ae"].n_iter_ >= models["ae"].max_iter else ""
    print(f"{tag:16s} {len(cols):2d} feats | AE n_iter_={models['ae'].n_iter_}{ae_note} "
          f"| {time.time()-t:,.0f}s")
    del X, models, S
print(f"\nall refits done in {(time.time()-T0)/60:.1f} min")

no_access_level  58 feats | AE n_iter_=40 (iteration-capped, not converged) | 107s
no_cat_freq      57 feats | AE n_iter_=40 (iteration-capped, not converged) | 95s
no_hours         56 feats | AE n_iter_=40 (iteration-capped, not converged) | 95s
no_peer_z        47 feats | AE n_iter_=40 (iteration-capped, not converged) | 89s

all refits done in 6.5 min


## 4. Check against the saved as-run ablations

Compare each refitted variant with `ablation_validation.csv` under the saved detector, as-run labels and common budget. The check requires exact agreement in false-positive counts and differences below `1e-9` for average precision and recall.

The notebook stops if any variant fails. Passing establishes agreement for these checked quantities, not byte-for-byte identity of every fitted object or prediction. The original printed wording is retained in the saved execution record.


In [5]:
# ============================================================
# 4. Reproduction gate vs committed ablation_validation.csv
# ============================================================
from sklearn.metrics import average_precision_score

committed = pd.read_csv(P_ABL).set_index("variant")
y_asrun = Y["as_run"]["val"]
gate = []
for tag in ABLATE:
    sv = SV[tag][DET]
    m  = mu.metrics_from_counts(*mu.counts(y_asrun, mu.alerts_at_budget(sv, K)))
    pr = average_precision_score(y_asrun, sv)
    r  = committed.loc[tag]
    ok = (abs(pr - r.val_pr_auc) < 1e-9) and (m["fp"] == int(r.val_fp_at_k)) \
         and (abs(m["recall"] - r.val_recall_at_k) < 1e-9)
    gate.append({"variant": tag, "refit_pr_auc": pr, "committed_pr_auc": r.val_pr_auc,
                 "refit_fp": m["fp"], "committed_fp": int(r.val_fp_at_k),
                 "refit_recall": m["recall"], "committed_recall": r.val_recall_at_k,
                 "reproduced": bool(ok)})
gate = pd.DataFrame(gate)
gate.to_csv(f"{WORK}/ablation_reproduction_gate.csv", index=False)
print(gate.to_string(index=False))
assert gate["reproduced"].all(), (
    "Refit did NOT reproduce the committed ablation table. Do not report the corrected "
    "figures below — investigate library-version drift or input-version mismatch first.")
print("\nREPRODUCTION GATE PASSED - refit is byte-faithful to the committed run.")

        variant  refit_pr_auc  committed_pr_auc  refit_fp  committed_fp  refit_recall  committed_recall  reproduced
no_access_level      0.000619          0.000619      4056          4056         0.100             0.100        True
    no_cat_freq      0.000764          0.000764      4054          4054         0.150             0.150        True
       no_hours      0.000805          0.000805      4055          4055         0.125             0.125        True
      no_peer_z      0.000904          0.000904      4054          4054         0.150             0.150        True

REPRODUCTION GATE PASSED - refit is byte-faithful to the committed run.


## 5. Report both label views and all detector families

Evaluate the reduced representations under the as-run and corrected labels. Read `M2_full` and `M0_ref` from the committed scores rather than fitting new reference models.

The as-run columns allow comparison with the original table; the corrected columns show the effect of the documented attribution change. Differences between variants are exploratory and do not select a new primary representation from the test data.


In [6]:
# ============================================================
# 5. Full ablation table: variant x detector x label set
# ============================================================
scores_all = pd.read_parquet(P_SCORES)
scores_all["user"] = scores_all["user"].astype(str)
scores_all["day"]  = pd.to_datetime(scores_all["day"])
val_committed = scores_all[scores_all.partition == "val"].reset_index(drop=True)

REF = {"M2_full": "M2", "M0_ref": "M0"}
rows = []
for label_set, yy in [(n, Y[n]["val"]) for n in ("as_run", "corrected")]:
    for det in DETECTORS:
        for tag, cfg in REF.items():
            sv = val_committed[f"{cfg}_{det}"].to_numpy()
            m  = mu.metrics_from_counts(*mu.counts(yy, mu.alerts_at_budget(sv, K)))
            rows.append({"label_set": label_set, "detector": det, "variant": tag,
                         "n_features": len(M0_FEATURES) + (len(CTX_COLS) if cfg == "M2" else 0),
                         "val_pr_auc": average_precision_score(yy, sv),
                         "val_recall_at_k": m["recall"], "val_fp_at_k": m["fp"],
                         "val_tp_at_k": m["tp"], "refitted": False})
        for tag, drop in ABLATE.items():
            sv = SV[tag][det]
            m  = mu.metrics_from_counts(*mu.counts(yy, mu.alerts_at_budget(sv, K)))
            rows.append({"label_set": label_set, "detector": det, "variant": tag,
                         "n_features": len(M0_FEATURES) + len(CTX_COLS) - len(drop),
                         "val_pr_auc": average_precision_score(yy, sv),
                         "val_recall_at_k": m["recall"], "val_fp_at_k": m["fp"],
                         "val_tp_at_k": m["tp"], "refitted": True})
abl = pd.DataFrame(rows)
abl.to_csv(f"{WORK}/ablation_corrected.csv", index=False)

order = ["M0_ref", "M2_full", "no_access_level", "no_cat_freq", "no_hours", "no_peer_z"]
for label_set in ("as_run", "corrected"):
    for det in DETECTORS:
        sub = (abl[(abl.label_set == label_set) & (abl.detector == det)]
               .set_index("variant").loc[order])
        print(f"\n--- labels={label_set}, detector={det}, k={K}% ---")
        print(sub[["n_features", "val_pr_auc", "val_recall_at_k",
                   "val_tp_at_k", "val_fp_at_k"]].to_string())
        best = sub["val_pr_auc"].idxmax()
        beats_full = sub.loc[best, "val_pr_auc"] > sub.loc["M2_full", "val_pr_auc"]
        print(f"    highest PR-AUC: {best}"
              + ("  (an ablated variant outranks the full curated model)" if beats_full and best != "M2_full" else ""))


--- labels=as_run, detector=if, k=1.0% ---
                 n_features  val_pr_auc  val_recall_at_k  val_tp_at_k  val_fp_at_k
variant                                                                           
M0_ref                   41    0.000836            0.125            5         4055
M2_full                  59    0.000667            0.150            6         4054
no_access_level          58    0.000619            0.100            4         4056
no_cat_freq              57    0.000764            0.150            6         4054
no_hours                 56    0.000805            0.125            5         4055
no_peer_z                47    0.000904            0.150            6         4054
    highest PR-AUC: no_peer_z  (an ablated variant outranks the full curated model)

--- labels=as_run, detector=ae, k=1.0% ---
                 n_features  val_pr_auc  val_recall_at_k  val_tp_at_k  val_fp_at_k
variant                                                                          

## 6. Save the ablation summary and audit

Save the complete comparison, the reproduction-check table and a summary of the observed orderings. Read average precision separately from the fixed-budget detection counts: a ranking improvement need not produce more detections at one budget.

No new test-set hypothesis decision is made here.


In [7]:
# ============================================================
# 6. Summary + audit
# ============================================================
def rank_note(label_set, det):
    sub = abl[(abl.label_set == label_set) & (abl.detector == det)]
    return {"best_variant": str(sub.loc[sub.val_pr_auc.idxmax(), "variant"]),
            "m2_full_beats_m0": bool(
                sub.loc[sub.variant == "M2_full", "val_pr_auc"].iloc[0] >
                sub.loc[sub.variant == "M0_ref",  "val_pr_auc"].iloc[0])}

summary = {
    "utils_sha256_16": UTILS_SHA,
    "frozen_policy": {"detector": DET, "k_pct": K},
    "masqueraded_supervisor": sorted(MASQ),
    "val_positives": {n: int(Y[n]["val"].sum()) for n in Y},
    "ae_n_iter_by_variant": AE_ITERS,
    "ae_iteration_capped": bool(all(v >= 40 for v in AE_ITERS.values())),
    "reproduction_gate_passed": bool(gate["reproduced"].all()),
    "ranking": {f"{ls}|{det}": rank_note(ls, det)
                for ls in ("as_run", "corrected") for det in DETECTORS},
    "seed": SEED,
}
json.dump(summary, open(f"{WORK}/ablation_summary.json", "w"), indent=2, default=str)

print("CORRECTED-LABEL ABLATION AUDIT")
print("-" * 70)
print(f"1. All fitting/scoring via NB4 modelling_utils.py verbatim (sha {UTILS_SHA}), seed={SEED}.")
print(f"2. Refit reproduces committed ablation_validation.csv exactly: {summary['reproduction_gate_passed']}")
print(f"3. M2_full / M0_ref reference rows read from committed scores - not refitted.")
print(f"4. Validation partition only; the test partition is never touched here.")
print(f"5. Autoencoder iteration-capped at max_iter in every variant: "
      f"{summary['ae_iteration_capped']} (n_iter_ = {AE_ITERS}).")
print(f"6. Corrected labels: {summary['val_positives']['corrected']} validation positives "
      f"vs {summary['val_positives']['as_run']} as-run.")
print("-" * 70)
print(f"\ntotal runtime: {(time.time()-T0)/60:.1f} min")
for f in sorted(os.listdir(WORK)):
    p = f"{WORK}/{f}"
    if os.path.isfile(p):
        print(f"  {f:40s} {os.path.getsize(p)/1e6:8,.3f} MB")

CORRECTED-LABEL ABLATION AUDIT
----------------------------------------------------------------------
1. All fitting/scoring via NB4 modelling_utils.py verbatim (sha 147d19b0977c79e8), seed=42.
2. Refit reproduces committed ablation_validation.csv exactly: True
3. M2_full / M0_ref reference rows read from committed scores - not refitted.
4. Validation partition only; the test partition is never touched here.
5. Autoencoder iteration-capped at max_iter in every variant: True (n_iter_ = {'no_access_level': 40, 'no_cat_freq': 40, 'no_hours': 40, 'no_peer_z': 40}).
6. Corrected labels: 9 validation positives vs 40 as-run.
----------------------------------------------------------------------

total runtime: 6.7 min
  __notebook__.ipynb                          0.034 MB
  ablation_corrected.csv                      0.003 MB
  ablation_reproduction_gate.csv              0.000 MB
  ablation_summary.json                       0.001 MB
